# Proyecto ML - Parte 2: Deep Learning y Aumentación de Datos

Este notebook implementa la segunda parte del proyecto: clasificación de textos por década usando una arquitectura profunda con TensorFlow/Keras, aumentación de datos y transferencia desde un modelo preentrenado/congelado.

La salida para Kaggle se guarda en `submission_parte2.csv` con el formato exacto `id,answer`. El modelo se guarda en formato Keras como `modelo_parte2.keras` y la correspondencia de etiquetas en `modelo_parte2_labels.json`.


In [1]:
import json
import os
import random
from pathlib import Path

# Evita sobreuso de hilos en CPU y hace más estable la ejecución local/Kaggle.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)



KeyboardInterrupt



## 1. Carga de datos y rutas

`eval.csv` se usa exclusivamente para generar predicciones finales. No se usa para entrenamiento, validación, aumentación ni ajuste de hiperparámetros.


In [ ]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "Data" / "train.csv").exists():
    BASE_DIR = BASE_DIR / "Proyecto-ML"

TRAIN_PATH = BASE_DIR / "Data" / "train.csv"
EVAL_PATH = BASE_DIR / "Data" / "eval.csv"
TEACHER_MODEL_PATH = BASE_DIR / "modelo_parte1.joblib"
SUBMISSION_PATH = BASE_DIR / "submission_parte2.csv"
MODEL_PATH = BASE_DIR / "modelo_parte2.keras"
LABELS_PATH = BASE_DIR / "modelo_parte2_labels.json"

train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

print(f"train.csv: {train_df.shape}")
print(f"eval.csv: {eval_df.shape}")
print("Columnas train:", train_df.columns.tolist())
print("Columnas eval:", eval_df.columns.tolist())
display(train_df.head())
display(eval_df.head())


## 2. Limpieza mínima y codificación de etiquetas

La limpieza elimina solo filas inválidas. No se corrige ortografía antigua ni se normaliza manualmente el texto, porque esas señales pueden ayudar a distinguir décadas.


In [ ]:
expected_train_cols = {"text", "decade"}
expected_eval_cols = {"id", "text"}
assert expected_train_cols.issubset(train_df.columns), "train.csv debe contener text y decade"
assert expected_eval_cols.issubset(eval_df.columns), "eval.csv debe contener id y text"

quality_summary = pd.DataFrame({
    "conjunto": ["train", "eval"],
    "filas": [len(train_df), len(eval_df)],
    "textos_nulos": [train_df["text"].isna().sum(), eval_df["text"].isna().sum()],
    "textos_vacios": [train_df["text"].fillna("").str.strip().eq("").sum(), eval_df["text"].fillna("").str.strip().eq("").sum()],
    "etiquetas_o_ids_nulos": [train_df["decade"].isna().sum(), eval_df["id"].isna().sum()],
    "duplicados_exactos": [train_df.duplicated().sum(), eval_df.duplicated().sum()],
})
display(quality_summary)

train_clean_df = train_df.dropna(subset=["text", "decade"]).copy()
train_clean_df["text"] = train_clean_df["text"].astype(str)
train_clean_df = train_clean_df[train_clean_df["text"].str.strip().ne("")]
train_clean_df = train_clean_df.drop_duplicates().reset_index(drop=True)

eval_model_df = eval_df.copy()
eval_model_df["text"] = eval_model_df["text"].fillna("").astype(str)

classes = np.array(sorted(train_clean_df["decade"].astype(int).unique()))
class_to_index = {int(label): int(i) for i, label in enumerate(classes)}
index_to_class = {int(i): int(label) for i, label in enumerate(classes)}
train_clean_df["label_id"] = train_clean_df["decade"].astype(int).map(class_to_index).astype(int)

print("Datos válidos de entrenamiento:", train_clean_df.shape)
print("Número de clases:", len(classes))
print("Rango de clases:", classes.min(), classes.max())


## 3. Transferencia desde un modelo preentrenado/congelado

Para cumplir la exigencia de transferencia/modelo preentrenado sin usar `eval.csv`, se carga el mejor modelo clásico ya entrenado de Parte 1 (`modelo_parte1.joblib`) como *teacher* congelado. Sus probabilidades por clase se pasan como una entrada auxiliar a la red neuronal Keras; el teacher no se reentrena en esta parte.

Si se decide usar embeddings externos como fastText/BETO en una iteración posterior, se deben añadir las referencias y licencias en el documento adicional exigido por el enunciado.


In [ ]:
assert TEACHER_MODEL_PATH.exists(), (
    "No se encontró modelo_parte1.joblib. Para esta implementación de transferencia, "
    "primero debe existir el modelo entrenado de Parte 1."
)

teacher_model = joblib.load(TEACHER_MODEL_PATH)
assert hasattr(teacher_model, "predict_proba"), "El modelo teacher debe exponer predict_proba."

teacher_classes = np.array([int(c) for c in teacher_model.classes_])
assert set(classes).issubset(set(teacher_classes)), "Las clases del teacher no cubren todas las clases de train.csv."

teacher_class_to_col = {int(label): int(i) for i, label in enumerate(teacher_classes)}
teacher_cols_for_project = [teacher_class_to_col[int(label)] for label in classes]


def teacher_proba_for(texts, batch_size=2048):
    parts = []
    texts = list(texts)
    for start in range(0, len(texts), batch_size):
        probs = teacher_model.predict_proba(texts[start:start + batch_size])
        parts.append(probs[:, teacher_cols_for_project])
    return np.vstack(parts).astype("float32")

print("Teacher cargado:", TEACHER_MODEL_PATH)
print("Clases transferidas:", len(teacher_cols_for_project))


## 4. Aumentación de datos

La aumentación se aplica únicamente al subconjunto local de entrenamiento. Se generan variantes sintéticas conservadoras mediante eliminación de palabras, intercambio de palabras adyacentes, ruido de caracteres y recortes parciales. No se usa `eval.csv` para construir ejemplos aumentados.


In [ ]:
def augment_text(text, rng):
    words = str(text).split()
    if len(words) < 8:
        return text

    op = rng.choice(["drop", "swap", "char_noise", "crop"])
    words_aug = words.copy()

    if op == "drop":
        drop_count = max(1, int(0.08 * len(words_aug)))
        drop_positions = set(rng.choice(len(words_aug), size=drop_count, replace=False))
        words_aug = [w for i, w in enumerate(words_aug) if i not in drop_positions]

    elif op == "swap" and len(words_aug) > 2:
        swaps = max(1, int(0.04 * len(words_aug)))
        for _ in range(swaps):
            i = int(rng.integers(0, len(words_aug) - 1))
            words_aug[i], words_aug[i + 1] = words_aug[i + 1], words_aug[i]

    elif op == "char_noise":
        idxs = rng.choice(len(words_aug), size=max(1, int(0.05 * len(words_aug))), replace=False)
        for idx in idxs:
            token = words_aug[int(idx)]
            if len(token) > 4:
                pos = int(rng.integers(1, len(token) - 1))
                words_aug[int(idx)] = token[:pos] + token[pos + 1:]

    elif op == "crop" and len(words_aug) > 20:
        keep = max(8, int(0.85 * len(words_aug)))
        start = int(rng.integers(0, len(words_aug) - keep + 1))
        words_aug = words_aug[start:start + keep]

    augmented = " ".join(words_aug).strip()
    return augmented if augmented else text


def build_augmented_frame(df, fraction=0.35, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n_aug = int(len(df) * fraction)
    sampled_idx = rng.choice(df.index.to_numpy(), size=n_aug, replace=False)
    sampled = df.loc[sampled_idx, ["text", "label_id", "decade"]].copy()
    sampled["text"] = [augment_text(text, rng) for text in sampled["text"]]
    sampled["is_augmented"] = 1

    original = df[["text", "label_id", "decade"]].copy()
    original["is_augmented"] = 0
    return pd.concat([original, sampled], ignore_index=True)


## 5. Partición local y matrices de entrada

La métrica principal es `accuracy`, igual que en Kaggle. También se reporta `f1_macro` para revisar el comportamiento entre clases.


In [ ]:
train_part_df, val_df = train_test_split(
    train_clean_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train_clean_df["label_id"],
)

train_aug_df = build_augmented_frame(train_part_df, fraction=0.35, seed=RANDOM_STATE)

X_train_text = train_aug_df["text"].to_numpy(dtype=object)
y_train = train_aug_df["label_id"].to_numpy(dtype="int32")
X_val_text = val_df["text"].to_numpy(dtype=object)
y_val = val_df["label_id"].to_numpy(dtype="int32")

X_train_teacher = teacher_proba_for(X_train_text)
X_val_teacher = teacher_proba_for(X_val_text)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(classes)),
    y=train_part_df["label_id"].to_numpy(),
)
class_weight = {int(i): float(w) for i, w in enumerate(class_weights_values)}

print("Entrenamiento original:", len(train_part_df))
print("Entrenamiento con aumentación:", len(train_aug_df))
print("Validación local:", len(val_df))
print("Entradas teacher train:", X_train_teacher.shape)


## 6. Modelo deep learning con Keras

La arquitectura tiene una rama de texto con `TextVectorization`, `Embedding`, `Conv1D` y pooling global. Esa representación se concatena con las probabilidades transferidas del teacher congelado y se clasifica con capas densas.


In [ ]:
MAX_TOKENS = 70000
SEQUENCE_LENGTH = 360
EMBEDDING_DIM = 128
BATCH_SIZE = 128
EPOCHS = 6

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train_text).batch(BATCH_SIZE))

text_input = keras.Input(shape=(1,), dtype=tf.string, name="text")
teacher_input = keras.Input(shape=(len(classes),), dtype=tf.float32, name="teacher_proba")

x = vectorizer(text_input)
x = layers.Embedding(MAX_TOKENS, EMBEDDING_DIM, mask_zero=False, name="token_embedding")(x)
x = layers.SpatialDropout1D(0.20)(x)
x = layers.Conv1D(160, 5, padding="same", activation="relu")(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.35)(x)

teacher_branch = layers.Dense(64, activation="relu", trainable=False, name="frozen_teacher_projection")(teacher_input)
combined = layers.Concatenate(name="text_teacher_concat")([x, teacher_branch])
combined = layers.Dense(128, activation="relu")(combined)
combined = layers.Dropout(0.25)(combined)
output = layers.Dense(len(classes), activation="softmax", name="decade")(combined)

model = keras.Model(inputs={"text": text_input, "teacher_proba": teacher_input}, outputs=output)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 7. Entrenamiento y evaluación local


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=2,
        restore_best_weights=True,
    )
]

history = model.fit(
    {"text": X_train_text, "teacher_proba": X_train_teacher},
    y_train,
    validation_data=({"text": X_val_text, "teacher_proba": X_val_teacher}, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

val_probs = model.predict({"text": X_val_text, "teacher_proba": X_val_teacher}, batch_size=BATCH_SIZE, verbose=0)
y_val_pred = val_probs.argmax(axis=1)

metrics = pd.DataFrame({
    "conjunto": ["validacion"],
    "accuracy": [accuracy_score(y_val, y_val_pred)],
    "f1_macro": [f1_score(y_val, y_val_pred, average="macro")],
})
display(metrics)
print(classification_report(y_val, y_val_pred, zero_division=0))


## 8. Entrenamiento final y archivo para Kaggle

Para el envío se entrena una instancia final con todos los datos etiquetados válidos y aumentación sintética. El archivo final se guarda sin índice extra en `submission_parte2.csv`.


In [ ]:
final_aug_df = build_augmented_frame(train_clean_df, fraction=0.35, seed=RANDOM_STATE + 1)
X_final_text = final_aug_df["text"].to_numpy(dtype=object)
y_final = final_aug_df["label_id"].to_numpy(dtype="int32")
X_final_teacher = teacher_proba_for(X_final_text)

final_vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)
final_vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_final_text).batch(BATCH_SIZE))

final_text_input = keras.Input(shape=(1,), dtype=tf.string, name="text")
final_teacher_input = keras.Input(shape=(len(classes),), dtype=tf.float32, name="teacher_proba")

x = final_vectorizer(final_text_input)
x = layers.Embedding(MAX_TOKENS, EMBEDDING_DIM, mask_zero=False, name="token_embedding")(x)
x = layers.SpatialDropout1D(0.20)(x)
x = layers.Conv1D(160, 5, padding="same", activation="relu")(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.35)(x)

teacher_branch = layers.Dense(64, activation="relu", trainable=False, name="frozen_teacher_projection")(final_teacher_input)
combined = layers.Concatenate(name="text_teacher_concat")([x, teacher_branch])
combined = layers.Dense(128, activation="relu")(combined)
combined = layers.Dropout(0.25)(combined)
final_output = layers.Dense(len(classes), activation="softmax", name="decade")(combined)

final_model = keras.Model(inputs={"text": final_text_input, "teacher_proba": final_teacher_input}, outputs=final_output)
final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

final_model.fit(
    {"text": X_final_text, "teacher_proba": X_final_teacher},
    y_final,
    epochs=max(2, len(history.history["loss"])),
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    verbose=1,
)

X_eval_text = eval_model_df["text"].to_numpy(dtype=object)
X_eval_teacher = teacher_proba_for(X_eval_text)
eval_probs = final_model.predict({"text": X_eval_text, "teacher_proba": X_eval_teacher}, batch_size=BATCH_SIZE, verbose=0)
eval_label_ids = eval_probs.argmax(axis=1)
eval_predictions = np.array([index_to_class[int(i)] for i in eval_label_ids], dtype=int)

submission_df = pd.DataFrame({
    "id": eval_model_df["id"],
    "answer": eval_predictions,
})

assert list(submission_df.columns) == ["id", "answer"]
assert len(submission_df) == len(eval_df)
assert submission_df.isna().sum().sum() == 0
assert set(submission_df["answer"]).issubset(set(classes))

submission_df.to_csv(SUBMISSION_PATH, index=False)
final_model.save(MODEL_PATH)
LABELS_PATH.write_text(json.dumps({"index_to_class": index_to_class}, indent=2), encoding="utf-8")

print(f"Archivo Kaggle guardado en: {SUBMISSION_PATH}")
print(f"Modelo Keras guardado en: {MODEL_PATH}")
print(f"Etiquetas guardadas en: {LABELS_PATH}")
print(f"Filas submission: {len(submission_df)}")
display(submission_df.head())


## 9. Entregables y notas de cumplimiento

Para esta primera implementación de Parte 2, la implementación debe permanecer en `Proyecto_ML_Parte2.ipynb`, porque el enunciado pide un Jupyter Notebook usado para implementar, entrenar y evaluar el modelo profundo. No es necesario crear un notebook adicional.

Archivos generados:

- `submission_parte2.csv`: archivo para enviar a Kaggle.
- `modelo_parte2.keras`: modelo entrenado con TensorFlow/Keras.
- `modelo_parte2_labels.json`: correspondencia entre índices internos y décadas.

No se usan datos externos en esta versión. Por eso el documento de referencias externas puede indicar que esta primera implementación no incorporó fuentes externas; si más adelante se agregan embeddings o corpus abiertos, habrá que documentar fuente, enlace y licencia explícita.
